# 00 — Verify Connection

Confirms the three things the rest of the demo depends on:

1. **Spark / Databricks Connect** session is reachable (only needed if `PERSIST_TO_UC=true`)
2. **rostr.cc** login works — POSTs to `/v1/auth/rostr` and caches the session cookie
3. **Google Sheet** is readable — reads the artists tab and reports current state

Run this first. `01_enrich_artists` assumes all three checks have passed.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

PERSIST_TO_UC = os.getenv('PERSIST_TO_UC', 'false').lower() == 'true'
print(f'PERSIST_TO_UC = {PERSIST_TO_UC}')

## 1. Spark / UC (skip if PERSIST_TO_UC=false)

Only needed when you want the optional Bronze/Silver audit trail. Otherwise we don't touch Databricks at runtime — the demo runs sheet → rostr → sheet.

In [ ]:
if PERSIST_TO_UC:
    try:
        spark
    except NameError:
        from databricks.connect import DatabricksSession
        spark = DatabricksSession.builder.serverless().getOrCreate()

    UC_CATALOG = os.getenv('UC_CATALOG', 'alexander_booth')
    UC_SCHEMA  = os.getenv('UC_SCHEMA',  'rostr_artist_enrichment')
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}')
    spark.sql(f'CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}.raw_data')
    print(f'Spark version: {spark.version}')
    spark.sql('SELECT current_user(), current_timestamp()').show(truncate=False)
    print(f'  schema ready: {UC_CATALOG}.{UC_SCHEMA}')
    print(f'  volume ready: /Volumes/{UC_CATALOG}/{UC_SCHEMA}/raw_data')
else:
    print('PERSIST_TO_UC=false — skipping Spark / UC checks.')

## 2. rostr.cc login

POSTs `email + password` to `/v1/auth/rostr` and caches the resulting `rack.session` cookie
to `ROSTR_COOKIE_PATH` so subsequent runs reuse it. The token cookie expires after a long
period (years from the API response) — re-login is automatic if it ever 401s.

In [ ]:
from rostr_client import RostrClient

rostr = RostrClient(
    username=os.environ['ROSTR_USERNAME'],
    password=os.environ['ROSTR_PASSWORD'],
    cookie_path=os.getenv('ROSTR_COOKIE_PATH') or None,
)
rostr.authenticate()

# Probe a known artist so we know the API is reachable.
team = rostr.lookup_artist_team('Olivia Rodrigo')
print(f'rostrId      : {team.rostr_id}')
print(f'name         : {team.name}')
print(f'AGENCY rows  : {len(team.by_role("AGENCY"))}')
print(f'MGMT rows    : {len(team.by_role("MANAGEMENT"))}')
for c in team.contacts:
    print(f'  - {c.role:<10s} {c.company:<25s} {c.person_name or "":<22s} {c.person_title or "":<10s} {c.person_email or ""}')

## 3. Google Sheet read

Local dev uses `gcloud auth application-default login` (your laptop's identity).
In a Databricks job, set `GOOGLE_AUTH_MODE=service` and provide a service-account JSON.

In [ ]:
from sheets_client import SheetsClient

sheets = SheetsClient(
    sheet_id=os.environ['GOOGLE_SHEET_ID'],
    tab=os.getenv('GOOGLE_SHEET_TAB', 'Sheet1'),
    auth_mode=os.getenv('GOOGLE_AUTH_MODE', 'adc'),
    service_account_path=os.getenv('GOOGLE_APPLICATION_CREDENTIALS') or None,
    quota_project=os.getenv('GOOGLE_QUOTA_PROJECT'),
)
sheets.ensure_canonical_header()
rows = sheets.read_artists()

needs = [r for r in rows if r.needs_enrichment()]
print(f'Rows total           : {len(rows)}')
print(f'Rows needing enrich  : {len(needs)}')
for r in rows:
    flag = 'BLANK' if r.needs_enrichment() else 'ok   '
    print(f'  [{flag}] {r.artist}')

All three checks passing → you're ready to run `01_enrich_artists`.